# Дообучение deepvk/USER-bge-m3 на парах P1 и кодирование корпуса (Kaggle)
Те же пары, что для e5-base (`src/make_pairs.py`, только история P1 с прореживанием), но **без префиксов**: у USER-bge-m3 их нет.
CachedMultipleNegativesRankingLoss, сэмплер NO_DUPLICATES, lr 1e-5, seed 42, те же версии библиотек, что локально.
После 20 шагов в лог пишется скорость и прогноз эпохи. Если обучение не укладывается в `TIME_BUDGET_H`, оно останавливается раньше
(это записывается в train_log.json), чтобы осталось время на кодирование корпуса.
После обучения тексты локального корпуса (`items_text.parquet`, включает весь benchmark_items) кодируются и сохраняются в Output.

In [ ]:
# Те же версии, что локально (requirements-lock.txt)
!pip install -q sentence-transformers==6.1.0 transformers==5.17.0 datasets==5.0.1 accelerate==1.15.0

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # одна GPU: CachedMNRL с DataParallel ведёт себя плохо
os.environ["TOKENIZERS_PARALLELISM"] = "false"

LOCAL_SMOKE = os.environ.get("LOCAL_SMOKE") == "1"   # проверка ноутбука на маке, на Kaggle не трогать
MODEL = "intfloat/multilingual-e5-small" if LOCAL_SMOKE else "deepvk/USER-bge-m3"
MAX_LEN = 128                      # как у dense-каналов на инференсе
BATCH = 16 if LOCAL_SMOKE else 512 # чем больше батч, тем больше in-batch негативов
MINI_BATCH = 8 if LOCAL_SMOKE else 32  # bge-m3 (24 слоя) тяжелее e5-base: мини-батч меньше
LR, WARMUP, SEED = 1e-5, 0.1, 42
TIME_BUDGET_H = 9.5                # из 12 ч сессии: остальное на кодирование корпуса и запись Output
MAX_STEPS = 2 if LOCAL_SMOKE else -1

import glob
def find(name):
    pat = f"artifacts/kaggle/{name}" if LOCAL_SMOKE else f"/kaggle/input/**/{name}"
    return glob.glob(pat, recursive=True)[0]
PAIRS, ITEMS = find("pairs_p1.parquet"), find("items_text.parquet")
WORK = "/tmp/bge_smoke" if LOCAL_SMOKE else "/kaggle/working"
os.makedirs(WORK, exist_ok=True)
print(PAIRS, ITEMS, WORK)

In [ ]:
import json, time
import numpy as np, pandas as pd
import torch, transformers, datasets, sentence_transformers
from transformers import TrainerCallback
from sentence_transformers import SentenceTransformer, SentenceTransformerTrainer
from sentence_transformers.sentence_transformer.training_args import SentenceTransformerTrainingArguments, BatchSamplers
from sentence_transformers.sentence_transformer.losses import CachedMultipleNegativesRankingLoss

assert sentence_transformers.__version__ == "6.1.0", sentence_transformers.__version__
print("st", sentence_transformers.__version__, "| transformers", transformers.__version__, "| torch", torch.__version__,
      "| cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
transformers.set_seed(SEED)

In [ ]:
# Пары те же, что для e5; снимаем префиксы E5 (у bge-m3 их нет)
train = datasets.Dataset.from_parquet(PAIRS)
if LOCAL_SMOKE:
    train = train.select(range(200))
train = train.map(lambda r: {"anchor": r["anchor"].removeprefix("query: "),
                             "positive": r["positive"].removeprefix("passage: ")})
assert not any(a.startswith("query: ") for a in train["anchor"][:1000])
assert not any(p.startswith("passage: ") for p in train["positive"][:1000])
print(train)
print(train[0])

In [ ]:
class SpeedAndBudget(TrainerCallback):
    """Лог скорости после 20 шагов и остановка по бюджету времени."""
    def __init__(self, budget_h):
        self.budget_s, self.t0, self.stopped_by_budget, self.sec_per_step = budget_h * 3600, None, False, None
    def on_step_begin(self, args, state, control, **kw):
        if self.t0 is None:
            self.t0 = time.time()
    def on_step_end(self, args, state, control, **kw):
        el = time.time() - self.t0
        if state.global_step == 20:
            self.sec_per_step = el / 20
            print(f"СКОРОСТЬ: {self.sec_per_step:.2f} с/шаг, прогноз эпохи ({state.max_steps} шагов): "
                  f"{self.sec_per_step * state.max_steps / 3600:.2f} ч, бюджет {self.budget_s / 3600:.1f} ч", flush=True)
        if el > self.budget_s:
            self.stopped_by_budget = True
            control.should_training_stop = True
        return control

model = SentenceTransformer(MODEL)
model.max_seq_length = MAX_LEN
loss = CachedMultipleNegativesRankingLoss(model, mini_batch_size=MINI_BATCH)
args = SentenceTransformerTrainingArguments(
    output_dir=f"{WORK}/ckpt",
    num_train_epochs=1,
    max_steps=MAX_STEPS,
    per_device_train_batch_size=BATCH,
    learning_rate=LR,
    warmup_steps=WARMUP,                # в transformers 5 доля прогрева задаётся float в warmup_steps
    fp16=torch.cuda.is_available(),     # T4/P100 не умеют bf16
    use_cpu=LOCAL_SMOKE,                # на MPS sdpa не поддерживает dropout; на Kaggle не влияет
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    dataloader_drop_last=True,
    seed=SEED,
    data_seed=SEED,
    logging_steps=10,
    save_strategy="no",                 # чекпоинты не нужны: веса пишем в конце, Output меньше
    report_to="none",
)
cb = SpeedAndBudget(TIME_BUDGET_H)
trainer = SentenceTransformerTrainer(model=model, args=args, train_dataset=train, loss=loss, callbacks=[cb])

In [ ]:
t0 = time.time()
trainer.train()
dt = time.time() - t0
steps, steps_epoch = trainer.state.global_step, len(train) // BATCH
print(f"{steps} шагов за {dt / 60:.1f} мин; эпоха {steps_epoch} шагов; остановлено по бюджету: {cb.stopped_by_budget}")
model.save(f"{WORK}/bge-m3-ft")
json.dump({"log_history": trainer.state.log_history, "sec_per_step": dt / max(steps, 1), "steps": steps,
           "steps_epoch": steps_epoch, "epoch_fraction": steps / steps_epoch, "stopped_by_budget": cb.stopped_by_budget,
           "model": MODEL, "batch": BATCH, "mini_batch": MINI_BATCH, "lr": LR, "seed": SEED, "max_len": MAX_LEN},
          open(f"{WORK}/train_log_bge.json", "w"), ensure_ascii=False, indent=1)

In [ ]:
# Кодирование локального корпуса (включает весь benchmark_items) без префикса.
# float16: вдвое меньше Output; локально переводится в float32, на косинус это не влияет.
items = pd.read_parquet(ITEMS)
if LOCAL_SMOKE:
    items = items.head(300)
t0 = time.time()
emb = model.encode(items["text"].tolist(), batch_size=256, normalize_embeddings=True,
                   convert_to_numpy=True, show_progress_bar=True)
print(f"закодировано {emb.shape} за {(time.time() - t0) / 60:.1f} мин")
np.savez(f"{WORK}/emb_bgem3ft.npz", ids=items["item_id"].to_numpy().astype(str), emb=emb.astype(np.float16))
print(sorted(os.listdir(WORK)))